In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os
from pathlib import Path

sc.settings.set_figure_params(dpi=300, frameon=False, facecolor='white')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.sans-serif'] = 'Arial'

# Path configuration (Masking absolute paths)
FILE_PATH = Path(os.getenv("SC_INTEGRATED_H5AD", "./data/processed/adata_integrated_cleaned.h5ad"))
SAVE_DIR_ZIKV = Path(os.getenv("SC_ZIKV_OUT", "./results/final_annotation/ZIKV_Mapping"))
SAVE_DIR_MOCK = Path(os.getenv("SC_MOCK_OUT", "./results/final_annotation/Figure1_Baseline_Clean"))

os.makedirs(SAVE_DIR_ZIKV, exist_ok=True)
os.makedirs(SAVE_DIR_MOCK, exist_ok=True)

# Load data
adata = sc.read_h5ad(FILE_PATH)
adata.obs_names_make_unique()

# 1. Pathogen Gene Configuration
virus_gene = 'ZIKV' 
if virus_gene not in adata.var_names:
    print(f"Error: Pathogen marker '{virus_gene}' not found in data")
    exit()

# 2. ZIKV Expression Pathology Map
custom_cmap_red = mcolors.LinearSegmentedColormap.from_list("sci_red", ["#E6E6E6", "#3817CC"])

print(f"Mapping viral gene expression target: '{virus_gene}'")
fig, ax = plt.subplots(figsize=(6, 5))

sc.pl.umap(
    adata, 
    color=virus_gene, 
    cmap=custom_cmap_red, 
    vmax=1.0,           
    vmin=0,             
    size=12.0,          
    sort_order=True,    
    frameon=False, 
    ax=ax, 
    show=False
)

for child in ax.get_children():
    if isinstance(child, plt.matplotlib.collections.PathCollection):
        child.set_rasterized(True)

ax.set_title(f"$\it{{{virus_gene}}}$ (ZIKV$^+$ Cells in Red)", fontsize=16, fontweight='bold')
save_path_zikv = SAVE_DIR_ZIKV / f"ZIKV_Mapping_Red_vmax1_size12.pdf"
fig.savefig(save_path_zikv, bbox_inches='tight', transparent=True)
plt.show()

# 3. SCI Baseline Map (Mock Backbone)
print("Generating standard gray-scale homeostatic MoCK baseline...")
is_mock = adata.obs['Sample'].str.startswith('MoCK')
is_virus_free = (adata[:, virus_gene].X == 0)

if hasattr(is_virus_free, "toarray"):
    is_virus_free = is_virus_free.toarray().flatten()

final_mask = is_mock & is_virus_free
final_count = final_mask.sum()

fig, ax = plt.subplots(figsize=(7, 7))

# View-mode slicing
sc.pl.umap(
    adata[final_mask], 
    color=None, 
    size=18, 
    alpha=1, 
    ax=ax,
    show=False
)

for child in ax.get_children():
    if isinstance(child, plt.matplotlib.collections.PathCollection):
        child.set_facecolor("#8C8B91") 
        child.set_rasterized(True)

ax.set_title(f'Homeostatic Baseline (MoCK, n={final_count:,})', fontsize=16, fontweight='bold')
save_path_mock = SAVE_DIR_MOCK / "Figure1_UMAP_MoCK_NoCopy_DarkGray.pdf"
fig.savefig(save_path_mock, bbox_inches='tight', transparent=True)
plt.show()
print("Baseline plot complete")